# Notebook 01 – Làm sạch dữ liệu và tạo 3 bản dữ liệu
Notebook này **không tự viết logic làm sạch**. Nó gọi bộ làm sạch dùng chung trong `src/sparkmind/cleaning.py`, cùng bộ mà server sẽ dùng khi có form mới. Nhờ vậy số liệu ở notebook và ở web luôn giống nhau.

**Cách dùng:** bấm *Run All*. Chọn nguồn dữ liệu ở ô 1 (`NGUON`).

## 1. Chọn nguồn dữ liệu
- `'excel'`: đọc file `data/raw/data_raw.xlsx` (không cần MongoDB, dùng để thử).
- `'mongo'`: đọc collection `survey_form1` từ MongoDB Atlas (cần file `.env`).

In [1]:
import sys; sys.path.append('../src')
import pandas as pd
from sparkmind import config
from sparkmind.cleaning import clean, lay_ban, bang_loc_form
from sparkmind.loaders import doc_excel, doc_goc_mongo
from sparkmind.db import lay_db

NGUON = 'excel'   # đổi thành 'mongo' khi dữ liệu đã nằm trong MongoDB
raw = doc_excel() if NGUON == 'excel' else doc_goc_mongo(lay_db())
print('Số form gốc:', len(raw))

Số form gốc: 460


## 2. Làm sạch (gắn cờ, không xóa form nào)

In [2]:
sach = clean(raw)
sach[['dung_ai','co_mot_muc_12','co_mot_muc_C','co_mot_muc_B','co_mau_thuan']].sum()

dung_ai          432
co_mot_muc_12     75
co_mot_muc_C     133
co_mot_muc_B      91
co_mau_thuan     144
dtype: int64

## 3. Bảng T2: số form còn lại ở từng bước

In [3]:
t2 = bang_loc_form(sach); t2

,buoc,con_lai,bi_loai
0,Tổng số form,460,0
1,Chỉ giữ người dùng AI,432,28
2,Bỏ form chọn một mức cho cả 12 câu (bản chính),361,71
3,Bỏ thêm form chọn một mức riêng cho B hoặc C (...,297,64
4,Bỏ thêm form mâu thuẫn (bản rất chặt),231,66


## 4. Số form theo đợt thu

In [4]:
pd.crosstab(sach['dot'], [sach['in_chinh']], margins=True).rename(columns={True:'trong ban chinh', False:'ngoai ban chinh'})

in_chinh,ngoai ban chinh,trong ban chinh,All
dot,,,
dot1_rai_rac,8,33,41
dot2_16/9,49,208,257
dot3_4-5/11,42,120,162
All,99,361,460


## 5. Điểm sáng tạo và mức (bản chính)

In [5]:
chinh = lay_ban(sach, 'chinh')
print(chinh['diem_sang_tao'].describe().round(1))
chinh['muc'].value_counts()

count    361.0
mean      64.3
std       20.0
min        0.0
25%       50.0
50%       65.0
75%       80.0
max      100.0
Name: diem_sang_tao, dtype: float64


muc
Trung_binh    150
Thap          119
Cao            92
Name: count, dtype: int64

## 6. Lưu file ra `data/processed/`

In [6]:
config.PROCESSED.mkdir(parents=True, exist_ok=True)
for ban in ('chinh', 'chat', 'rat_chat'):
    b = lay_ban(sach, ban); b.to_csv(config.PROCESSED / f'data_{config.FILE_NAME[ban]}.csv', index=False, encoding='utf-8-sig')
    print(ban, len(b))
sach.to_excel(config.PROCESSED / 'data_with_flags.xlsx', index=False)
t2.to_csv(config.PROCESSED / 'T2_filter_steps.csv', index=False, encoding='utf-8-sig')

chinh 361
chat 297
rat_chat 231
